# CFD Geometry — Colab quick start

**NOTEBOOK_ID:** `cfd-colab-v5-stable-deps`

> **Important:** Open from GitHub (link below), **not** an old copy in Google Drive.
>
> [Open fresh notebook on GitHub](https://github.com/Omokayode/CFDGeometry/blob/main/notebooks/colab_quickstart.ipynb)

| Step | What you run |
|------|----------------|
| **1** | Install (next cell — must say `STEP 1`) |
| **2** | Map / draw extent |
| **3** | Build STLs |
| **4** | Plotly 3D preview |

If the cell after this title shows a **map** instead of `STEP 1 — Install`, you have the **wrong notebook**.


In [ ]:
# STEP 1 — Install (run this cell first)
import subprocess
import sys
from pathlib import Path

NOTEBOOK_ID = "cfd-colab-v5-stable-deps"
_GIT = "git+https://github.com/Omokayode/CFDGeometry.git@main"
_DEPS = (
    "ipywidgets>=7.6,<9",
    "ipyleaflet>=0.17",
    "jupyterlab_widgets>=1.0.5,<4",
    "plotly>=5.18",
    "nbformat>=4.2.0",
)
_STRATEGY = ("--upgrade-strategy", "only-if-needed")


def _pip(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "src" / "cfd_geometry" / "domain").is_dir():
            return p
    return here


def install_cfd_geometry() -> None:
    repo = find_repo_root()
    print("Repo root:", repo)
    if _in_colab():
        _pip(*_DEPS, *_STRATEGY)
        # Do not reinstall numpy/scipy (breaks Colab). Package only, then deps.
        _pip("--upgrade", "--no-cache-dir", "--no-deps", f"{_GIT}#egg=cfd-geometry")
        _pip("osmnx>=1.9", "requests>=2.28", *_STRATEGY)
        _pip(
            "geopandas>=0.14",
            "rasterio>=1.3",
            "trimesh>=4.0",
            "mapbox-earcut>=1.0",
            "scipy>=1.11",
            "shapely>=2.0",
            "pyproj>=3.6",
            "pandas>=2.0",
            *_STRATEGY,
        )
    elif (repo / "src" / "cfd_geometry").exists():
        _pip("-e", f"{repo}[notebook,download]")
    else:
        _pip(f"{_GIT}#egg=cfd-geometry[notebook,download]")


install_cfd_geometry()

from cfd_geometry.notebook import setup_colab_widgets

if setup_colab_widgets():
    print("Colab widgets enabled.")

import cfd_geometry
from cfd_geometry.domain import DomainConfig, build_domain  # noqa: F401

print("NOTEBOOK_ID", NOTEBOOK_ID)
print("cfd_geometry", cfd_geometry.__version__)
print("domain OK")


In [ ]:
# STEP 2 — Draw study extent (map)
import subprocess
import sys

try:
    import cfd_geometry
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--no-cache-dir", "--no-deps",
        "git+https://github.com/Omokayode/CFDGeometry.git@main#egg=cfd-geometry",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "osmnx>=1.9", "requests>=2.28",
        "--upgrade-strategy", "only-if-needed",
    ])
    import cfd_geometry

from cfd_geometry.notebook import setup_colab_widgets, select_extent

setup_colab_widgets()

PLACE = "Milwaukee, Wisconsin, USA"
CENTER = None

selector = select_extent(place=PLACE if CENTER is None else None, center=CENTER)
selector


In [ ]:
if selector.bbox is None:
    raise RuntimeError("Draw a rectangle, then click 'Use this extent'.")
bbox = selector.bbox
print(f"west={bbox.west:.6f} south={bbox.south:.6f} east={bbox.east:.6f} north={bbox.north:.6f}")


In [ ]:
# STEP 3 — Download OSM + extrude STLs
from pathlib import Path
from cfd_geometry.domain import DomainConfig, build_domain

result = build_domain(DomainConfig(
    output_dir=Path("data"),
    bbox=selector.bbox,
    run_download=True,
    download_layers=("buildings", "trees"),
    build_buildings=True,
    build_trees=True,
    height_source="composite",
))
result.stl_files


In [ ]:
from pathlib import Path
for p in sorted(Path("data/output").glob("*.stl")):
    print(p.name, f"{p.stat().st_size / 1024:.1f} KiB")


In [ ]:
# STEP 4 — Plotly 3D preview (last)
from cfd_geometry.notebook.visualize import plot_domain_stls
plot_domain_stls(result, layers=("buildings", "trees"), max_triangles=8000)
